# 08 — Visualization Storytelling

> **"Data is the raw material. Story is the finished product."**

---

Notebook cuối này kết hợp tất cả 7 concepts đã học:

| Notebook | Concept |
|----------|--------|
| 01 | Perception Control — signal vs noise, highlight, saliency |
| 02 | Narrative — turning points, events, regimes, captions |
| 03 | Context — benchmark, target, excess/deficit, comparison |
| 04 | Uncertainty — CI, PI, fan chart, forecast |
| 05 | Cognitive Load — direct label, declutter, typography |
| 06 | Process — waterfall, contribution, decomposition |
| 07 | Framing — bias recognition, honest chart checklist |

Mục tiêu: đi qua **full storytelling pipeline** từ raw data đến insight hoàn chỉnh.

### Nội dung notebook này:

1. **The Pipeline** — Raw Data → Signal → Context → Narrative → Story
2. **StoryChart** — High-level builder cho signal + background + events
3. **InsightChart** — Chart tập trung vào một insight duy nhất
4. **NarrativePipeline** — Full orchestration một call
5. **PipelineBuilder** — Fluent custom pipeline
6. **Multi-Chart Story** — Kể một câu chuyện qua nhiều chart
7. **Full Story Dashboard** — Từ data đến dashboard hoàn chỉnh

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

matplotlib.use('Agg')
%matplotlib inline

import sys, os
sys.path.insert(0, os.path.abspath('../src'))

# Core
from vizint.core import ChartBuilder, despine, set_grid, make_grid
from vizint.core.axes_utils import format_thousands, format_percent, set_tick_style

# Perception
from vizint.perception import (
    highlight_series, fade_series,
    focus_on_range, highlight_region,
)

# Context
from vizint.context import (
    add_reference_line, add_mean_line,
    add_target_line, add_benchmark_band,
    shade_excess, shade_deficit,
)

# Narrative
from vizint.narrative import (
    mark_turning_points,
    add_event_markers, add_timeline_band,
    shade_regimes, add_regime_labels,
    StoryFrame, add_narrative_caption,
)

# Annotation
from vizint.annotation import (
    label_last_point, label_max, label_min,
    add_note, add_caption, add_source_note,
    annotate_with_arrow, add_callout_arrow,
)

# Uncertainty
from vizint.uncertainty import (
    add_confidence_band, shade_forecast_period, add_forecast_region,
)

# Process
from vizint.process import waterfall_chart, contribution_chart

# Pipeline
from vizint.pipeline import (
    StoryChart, InsightChart,
    NarrativePipeline, PipelineBuilder,
)

# Styling
from vizint.styling import categorical_colors, apply_theme
from vizint.styling.typography import (
    TITLE_SIZE, SUBTITLE_SIZE, CAPTION_SIZE, ANNOTATION_SIZE
)

np.random.seed(2025)
print('Setup OK ✓')

## The Story: SaaS Company — 5-Year Journey

Dữ liệu giả lập: một SaaS company từ 2020 đến 2024.

**Câu chuyện cần kể:**
- 2020: COVID impact, revenue giảm
- 2021: Recovery + product launch mới
- 2022: Hyper-growth, vượt industry benchmark
- 2023: Growth chậm lại, market headwinds
- 2024: Stabilization, back on track

In [ ]:
# ── Time axis ──────────────────────────────────────────────────────────
T       = 60   # 60 months = 5 years
months  = np.arange(T)
dates   = pd.date_range('2020-01', periods=T, freq='ME')

# ── Primary: Company ARR ($K) ──────────────────────────────────────────
arr = np.concatenate([
    # 2020: starts strong, COVID dip mid-year
    np.linspace(800, 750, 12) + np.random.randn(12) * 15,
    # 2021: recovery + new product launch (month 18)
    np.linspace(760, 1100, 12) + np.random.randn(12) * 20,
    # 2022: hyper-growth
    np.linspace(1110, 1850, 12) + np.random.randn(12) * 25,
    # 2023: slowdown
    np.linspace(1860, 2050, 12) + np.random.randn(12) * 30,
    # 2024: stabilization
    np.linspace(2060, 2300, 12) + np.random.randn(12) * 20,
])

# ── Industry peers (background context) ───────────────────────────────
peers = {
    'Peer A': 900  + np.cumsum(np.random.randn(T) * 18 + 18),
    'Peer B': 1100 + np.cumsum(np.random.randn(T) * 22 + 12),
    'Peer C': 700  + np.cumsum(np.random.randn(T) * 15 + 20),
}

# ── Key events ────────────────────────────────────────────────────────
EVENTS = [
    (4,  'COVID'),
    (18, 'Product v2'),
    (36, 'Series B'),
    (48, 'Market\ downturn'),
]

# ── Regime breakpoints ────────────────────────────────────────────────
BREAKPOINTS  = [12, 24, 36, 48]
REGIME_NAMES = ['COVID Impact', 'Recovery', 'Hyper-Growth', 'Slowdown', 'Stabilization']

# ── Forecast (months 54-65) ───────────────────────────────────────────
T_fore       = 12
t_fore       = np.arange(T - 1, T - 1 + T_fore)
fore_central = arr[-1] + np.linspace(0, 300, T_fore)
fore_std     = np.linspace(20, 80, T_fore)
fore_lo      = fore_central - 1.96 * fore_std
fore_hi      = fore_central + 1.96 * fore_std

# ── Revenue bridge: 2023→2024 ─────────────────────────────────────────
bridge_labels = ['Jan 2024', 'New Logos', 'Expansion', 'Price',
                 'Churn', 'Contraction', 'Dec 2024']
bridge_values = [arr[48], +380, +210, +95, -120, -60, 0]
bridge_values[-1] = bridge_values[0] + sum(bridge_values[1:-1])

# ── Segment contribution ──────────────────────────────────────────────
segments      = ['Enterprise', 'Mid-Market', 'SMB', 'API/Dev', 'Other']
seg_contrib   = [44.2, 28.5, 15.3, 8.7, 3.3]

TARGET        = 2000   # ARR target
BENCHMARK_LO  = 1600
BENCHMARK_HI  = 1900

year_ticks  = [i for i in range(T) if i % 12 == 0]
year_labels = [str(2020 + i // 12) for i in year_ticks]

print(f'ARR range   : [{arr.min():.0f}, {arr.max():.0f}] $K')
print(f'Events      : {[e[1] for e in EVENTS]}')
print(f'Regimes     : {REGIME_NAMES}')

---
## 1. The Storytelling Pipeline

Trước khi dùng pipeline classes, hãy hiểu cấu trúc của nó:

```
Raw Data
   ↓
Signal Extraction    → highlight_series + fade_series
   ↓
Focus / Emphasis     → focus_on_range, highlight_region
   ↓
Context Injection    → benchmark, target, mean reference
   ↓
Narrative Layers     → regimes, events, turning points
   ↓
Annotation           → direct labels, arrows, callouts
   ↓
Caption / Takeaway   → add_narrative_caption
   ↓
Insight
```

Demo dưới đây xây dựng chart từng layer một để thấy rõ sự tích lũy.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), facecolor='white')
axes_flat = axes.flatten()

layer_titles = [
    'Layer 1: Raw data only',
    'Layer 2: + Signal vs Noise',
    'Layer 3: + Context (benchmark + target)',
    'Layer 4: + Narrative (regimes + events)',
    'Layer 5: + Annotation (labels + arrows)',
    'Layer 6: + Caption → Complete Story',
]

for layer, (ax, title) in enumerate(zip(axes_flat, layer_titles)):

    # Layer 1+: always plot all peers and company
    if layer == 0:
        for name, y in peers.items():
            ax.plot(months, y, linewidth=1.5, alpha=0.7, label=name)
        ax.plot(months, arr, linewidth=2.0, label='Our Company')
        ax.legend(frameon=False, fontsize=7, loc='upper left')

    if layer >= 1:
        for name, y in peers.items():
            fade_series(ax, months, y, alpha=0.18, color='#CBD5E1', linewidth=0.9)
        highlight_series(ax, months, arr, color='#2563EB', linewidth=2.5)

    if layer >= 2:
        add_benchmark_band(ax, BENCHMARK_LO, BENCHMARK_HI,
                           color='#FEF3C7', alpha=0.4,
                           label=f'Industry avg')
        add_target_line(ax, TARGET, color='#16A34A',
                        linewidth=1.2, linestyle='--', label='Target')
        shade_excess(ax, months, arr, baseline=TARGET,
                     color='#BBF7D0', alpha=0.3)
        shade_deficit(ax, months, arr, baseline=TARGET,
                      color='#FECACA', alpha=0.3)

    if layer >= 3:
        shade_regimes(ax, BREAKPOINTS, alpha=0.10)
        add_regime_labels(ax, BREAKPOINTS, REGIME_NAMES,
                          fontsize=7, color='#6B7280')
        add_event_markers(ax,
                          [e[0] for e in EVENTS],
                          labels=[e[1] for e in EVENTS],
                          color='#7C3AED', linewidth=1.0,
                          label_fontsize=7)

    if layer >= 4:
        label_last_point(ax, months, arr,
                         text=f'${arr[-1]:.0f}K', color='#2563EB', fontsize=8)
        mark_turning_points(ax, months, arr, order=6,
                            max_color='#16A34A', min_color='#DC2626',
                            show_labels=True, label_fontsize=7, size=50)

    if layer >= 5:
        add_narrative_caption(
            ax,
            'Despite COVID impact in 2020, ARR grew 3x by 2024, '
            'surpassing industry benchmark from mid-2022 onwards.',
            y=-0.18, fontsize=8,
        )

    ax.set_xticks(year_ticks)
    ax.set_xticklabels(year_labels, fontsize=8)
    ax.set_title(title, loc='left', fontsize=9, fontweight='bold')
    ax.set_ylabel('ARR ($K)', fontsize=8)
    format_thousands(ax, 'y')
    despine(ax); set_grid(ax)
    set_tick_style(ax, labelsize=8)

fig.suptitle('Visualization Storytelling Pipeline — 6 Layers',
             fontsize=13, fontweight='bold', color='#1a1a1a', y=1.01)
plt.tight_layout(pad=2.5)
plt.show()

---
## 2. StoryChart — Signal + Background + Events

`StoryChart` là high-level builder kết hợp:
- **Signal series** — highlighted
- **Background series** — faded context
- **Event markers**
- **Mean reference**
- **Narrative caption**

Fluent API: `.add_event().add_caption().add_mean_reference().build()`

In [ ]:
chart = (
    StoryChart(
        x=months,
        signal_series={'Our Company': arr},
        background_series=peers,
        title='ARR Growth — 2020 to 2024',
        subtitle='Signal vs industry peers  |  Key events annotated',
        figsize=(14, 5),
    )
    .add_event(4,  'COVID')
    .add_event(18, 'Product v2')
    .add_event(36, 'Series B')
    .add_event(48, 'Market downturn')
    .add_mean_reference(arr)
    .add_caption(
        'Our ARR grew 3x from 2020 to 2024, consistently outpacing peers '
        'after the Product v2 launch in early 2021.'
    )
    .build()
)

ax = chart.ax
ax.set_xticks(year_ticks)
ax.set_xticklabels(year_labels, fontsize=8)
ax.set_ylabel('ARR ($K)')
format_thousands(ax, 'y')

plt.tight_layout()
plt.show()

---
## 3. InsightChart — Tập trung vào một insight

`InsightChart` thiết kế cho trường hợp có **một insight cụ thể** cần highlight:
- Một giai đoạn quan trọng
- Một callout annotation
- Một reference line

Phù hợp cho executive summary, slide thuyết trình, hay email report.

In [ ]:
# Insight: ARR crossed the $2M target in month 53
target_cross_idx = int(np.where(arr >= TARGET)[0][0]) if np.any(arr >= TARGET) else T-1

chart = (
    InsightChart(
        x=months,
        y=arr,
        title='ARR crossed $2M target — 5 months ahead of plan',
        figsize=(13, 5),
        series_color='#2563EB',
    )
    .set_insight_region(xmin=target_cross_idx - 2, xmax=target_cross_idx + 4)
    .set_reference(TARGET)
    .set_callout(
        xy=(target_cross_idx, arr[target_cross_idx]),
        text=f'Crossed $2M\nMonth {target_cross_idx}',
        dx=0.08, dy=0.12,
    )
    .set_caption(
        'ARR surpassed the $2M annual recurring revenue target in month '
        f'{target_cross_idx}, driven by enterprise expansion and '
        'successful Series B deployment.'
    )
    .build()
)

ax = chart.ax
ax.set_xticks(year_ticks)
ax.set_xticklabels(year_labels, fontsize=8)
ax.set_ylabel('ARR ($K)')
format_thousands(ax, 'y')

plt.tight_layout()
plt.show()

---
## 4. NarrativePipeline — Full Orchestration

`NarrativePipeline` là pipeline đầy đủ nhất — tích hợp tất cả layers:

```
regime shading → background series → signal → mean ref
    → turning points → event markers → label last → caption
```

Một call `.run()` tạo ra chart hoàn chỉnh.

In [ ]:
chart = (
    NarrativePipeline(
        x=months,
        y=arr,
        title='SaaS ARR — 5-Year Growth Story',
        subtitle='2020–2024  |  Monthly ARR ($K)  |  With industry peers',
        figsize=(15, 5.5),
        signal_color='#2563EB',
    )
    .add_regimes(
        breakpoints=BREAKPOINTS,
        labels=REGIME_NAMES,
    )
    .add_background(peers)
    .add_events(EVENTS)
    .add_turning_points()
    .add_mean_reference()
    .add_caption(
        'After the COVID dip in 2020, ARR recovered strongly through '
        'Product v2 and Series B, achieving 3x growth by end of 2024. '
        'Growth decelerated in 2023 amid market headwinds but stabilized in 2024.'
    )
    .run()
)

ax = chart.ax
ax.set_xticks(year_ticks)
ax.set_xticklabels(year_labels, fontsize=8)
ax.set_ylabel('ARR ($K)')
format_thousands(ax, 'y')

plt.tight_layout()
plt.show()

---
## 5. PipelineBuilder — Custom Fluent Pipeline

`PipelineBuilder` cho phép compose bất kỳ rendering step nào theo thứ tự tùy chỉnh.
Mỗi `.step(lambda ax: ...)` nhận `ax` và render gì đó lên đó.

Dùng khi cần mix các functions từ nhiều modules khác nhau
mà các pipeline cao cấp hơn không hỗ trợ.

In [ ]:
# Custom pipeline: ARR + forecast + uncertainty
chart = (
    PipelineBuilder(
        figsize=(14, 5),
        title='ARR — Historical + 12-Month Forecast',
        subtitle='95% confidence interval  |  Shaded = forecast period',
    )
    # Step 1: faded peers
    .step(lambda ax: [fade_series(ax, months, y, alpha=0.15, color='#CBD5E1')
                      for y in peers.values()])
    # Step 2: highlight ARR
    .step(lambda ax: highlight_series(ax, months, arr,
                                      color='#1D4ED8', linewidth=2.5))
    # Step 3: target line
    .step(lambda ax: add_target_line(ax, TARGET, color='#16A34A',
                                     linewidth=1.2, linestyle='--',
                                     label='$2M target'))
    # Step 4: forecast shading
    .step(lambda ax: shade_forecast_period(
        ax, T - 1, color='#FAF5FF', alpha=0.5,
        linecolor='#7C3AED', linewidth=1.2))
    # Step 5: forecast region
    .step(lambda ax: add_forecast_region(
        ax, t_fore, fore_central, fore_lo, fore_hi,
        central_color='#7C3AED', band_color='#C4B5FD',
        central_linewidth=2.0, band_alpha=0.25,
        linestyle='--', label='Forecast'))
    # Step 6: label last historical point
    .step(lambda ax: label_last_point(
        ax, months, arr,
        text=f'${arr[-1]:.0f}K', color='#1D4ED8', fontsize=8))
    # Step 7: caption
    .step(lambda ax: add_narrative_caption(
        ax,
        'Forecast projects ARR reaching $2.6M by end of 2025 '
        'under base-case assumptions. Uncertainty band widens with horizon.',
        y=-0.15, fontsize=8))
    .build()
)

ax = chart.ax
xticks_ext = list(year_ticks) + [T - 1 + 11]
xlabels_ext = year_labels + ['2025']
ax.set_xticks(xticks_ext)
ax.set_xticklabels(xlabels_ext, fontsize=8)
ax.set_ylabel('ARR ($K)')
format_thousands(ax, 'y')
ax.legend(frameon=False, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

---
## 6. Multi-Chart Story — Một câu chuyện qua nhiều chart

Đôi khi một chart không đủ để kể câu chuyện hoàn chỉnh.
Một **story sequence** gồm nhiều chart, mỗi chart trả lời một câu hỏi:

1. **Overview** — What happened overall?
2. **Zoom In** — When did the key moment happen?
3. **Why** — What drove the change?
4. **What's Next** — Where are we going?

In [ ]:
fig = plt.figure(figsize=(16, 12), facecolor='white')
fig.suptitle('SaaS Company — Full Story in 4 Charts',
             fontsize=14, fontweight='bold', color='#1a1a1a', y=1.01)

# ── Chart 1: Overview ─────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 2, 1)
for y in peers.values():
    fade_series(ax1, months, y, alpha=0.15, color='#CBD5E1')
highlight_series(ax1, months, arr, color='#2563EB', linewidth=2.2)
shade_regimes(ax1, BREAKPOINTS, alpha=0.10)
label_last_point(ax1, months, arr, text=f'${arr[-1]:.0f}K', color='#2563EB', fontsize=8)
ax1.set_xticks(year_ticks); ax1.set_xticklabels(year_labels, fontsize=8)
ax1.set_title('① What happened? — 5-Year ARR vs Peers',
              loc='left', fontsize=10, fontweight='bold')
ax1.set_ylabel('ARR ($K)'); format_thousands(ax1, 'y')
despine(ax1); set_grid(ax1); set_tick_style(ax1, labelsize=8)
add_note(ax1, '→ 3x growth, outpaced peers from 2022',
         x=0.03, y=0.08, color='#2563EB', fontsize=8)

# ── Chart 2: Zoom into hyper-growth ──────────────────────────────────
ax2 = fig.add_subplot(2, 2, 2)
zoom_s, zoom_e = 12, 36
ax2.plot(months[zoom_s:zoom_e], arr[zoom_s:zoom_e],
         color='#2563EB', linewidth=2.5)
focus_on_range(ax2, 18, 30, highlight_color='#EFF6FF', alpha=0.5,
               border_color='#2563EB', border_linewidth=1.0)
add_event_markers(ax2, [18, 24], labels=['Product v2', 'ISO cert'],
                  color='#7C3AED', label_fontsize=7)
mark_turning_points(ax2, months[zoom_s:zoom_e],
                    arr[zoom_s:zoom_e], order=3,
                    max_color='#16A34A', min_color='#DC2626',
                    show_labels=True, label_fontsize=7, size=50)
ax2.set_title('② When? — Hyper-Growth Phase (2021–2022)',
              loc='left', fontsize=10, fontweight='bold')
ax2.set_xlabel('Month'); ax2.set_ylabel('ARR ($K)')
format_thousands(ax2, 'y')
despine(ax2); set_grid(ax2); set_tick_style(ax2, labelsize=8)
add_note(ax2, '→ Product v2 sparked 68% growth in 18 months',
         x=0.03, y=0.08, color='#7C3AED', fontsize=8)

# ── Chart 3: Why — revenue bridge ────────────────────────────────────
ax3 = fig.add_subplot(2, 2, 3)
waterfall_chart(
    ax3,
    labels=bridge_labels,
    values=bridge_values,
    positive_color='#16A34A', negative_color='#DC2626', total_color='#2563EB',
    total_indices=[0, 6],
    bar_width=0.55, show_values=True, value_fontsize=7,
)
ax3.set_title('③ Why? — 2024 ARR Bridge (Jan → Dec)',
              loc='left', fontsize=10, fontweight='bold')
ax3.set_ylabel('ARR ($K)'); format_thousands(ax3, 'y')
set_tick_style(ax3, labelsize=8)
add_note(ax3, '→ New logos + expansion offset churn comfortably',
         x=0.03, y=0.08, color='#16A34A', fontsize=8)

# ── Chart 4: What's next — forecast ──────────────────────────────────
ax4 = fig.add_subplot(2, 2, 4)
ax4.plot(months, arr, color='#1D4ED8', linewidth=2.2, label='Historical')
shade_forecast_period(ax4, T - 1, color='#FAF5FF', alpha=0.5,
                      linecolor='#7C3AED', linewidth=1.0)
add_forecast_region(ax4, t_fore, fore_central, fore_lo, fore_hi,
                    central_color='#7C3AED', band_color='#C4B5FD',
                    central_linewidth=1.8, band_alpha=0.25,
                    linestyle='--', label='Forecast')
add_target_line(ax4, TARGET, color='#16A34A', linewidth=1.0,
                linestyle='--', label='$2M target')
xticks_ext = list(year_ticks) + [T - 1 + 11]
ax4.set_xticks(xticks_ext)
ax4.set_xticklabels(year_labels + ['2025'], fontsize=8)
ax4.set_title('④ What\'s next? — 12-Month Forecast',
              loc='left', fontsize=10, fontweight='bold')
ax4.set_ylabel('ARR ($K)'); format_thousands(ax4, 'y')
ax4.legend(frameon=False, fontsize=7, loc='upper left')
despine(ax4); set_grid(ax4); set_tick_style(ax4, labelsize=8)
add_note(ax4, f'→ Forecast: ${fore_central[-1]:.0f}K by Dec 2025',
         x=0.03, y=0.08, color='#7C3AED', fontsize=8)

plt.tight_layout(pad=2.0)
plt.show()

---
## 7. Full Story Dashboard

Kết hợp tất cả trong một dashboard hoàn chỉnh:
- Header chart: narrative timeline toàn bộ
- Bottom row: contribution + waterfall + distribution

Đây là output cuối của toàn bộ framework.

In [ ]:
from vizint.process import contribution_chart
from vizint.comparison import compare_lines

fig = plt.figure(figsize=(18, 13), facecolor='white')
fig.suptitle(
    'SaaS Company — Annual Business Review 2024',
    fontsize=16, fontweight='bold', color='#1a1a1a', y=1.005,
)

# ── Top: Full narrative ARR chart ─────────────────────────────────────
ax_top = fig.add_axes([0.04, 0.55, 0.92, 0.40])

for y in peers.values():
    fade_series(ax_top, months, y, alpha=0.15, color='#CBD5E1', linewidth=0.8)

highlight_series(ax_top, months, arr, color='#2563EB', linewidth=2.5)
shade_regimes(ax_top, BREAKPOINTS, alpha=0.08)
add_regime_labels(ax_top, BREAKPOINTS, REGIME_NAMES, fontsize=8, color='#6B7280')
add_benchmark_band(ax_top, BENCHMARK_LO, BENCHMARK_HI,
                   color='#FEF3C7', alpha=0.4, label='Industry avg')
add_target_line(ax_top, TARGET, color='#16A34A', linewidth=1.2,
                linestyle='--', label='$2M target')
shade_excess(ax_top, months, arr, baseline=TARGET,
             color='#BBF7D0', alpha=0.3)
add_event_markers(ax_top, [e[0] for e in EVENTS],
                  labels=[e[1] for e in EVENTS],
                  color='#7C3AED', label_fontsize=7)
mark_turning_points(ax_top, months, arr, order=7,
                    max_color='#16A34A', min_color='#DC2626',
                    show_labels=True, label_fontsize=7, size=45)
label_last_point(ax_top, months, arr,
                 text=f'${arr[-1]:.0f}K', color='#2563EB', fontsize=9)
ax_top.set_xticks(year_ticks)
ax_top.set_xticklabels(year_labels, fontsize=9)
ax_top.set_title('Annual Recurring Revenue — 2020 to 2024',
                 loc='left', fontsize=TITLE_SIZE,
                 fontweight='bold', color='#1a1a1a')
ax_top.set_ylabel('ARR ($K)', fontsize=10)
format_thousands(ax_top, 'y')
ax_top.legend(frameon=False, fontsize=8, loc='upper left')
despine(ax_top); set_grid(ax_top)
add_narrative_caption(
    ax_top,
    'ARR grew 3x from 2020 to 2024, surpassing the $2M target in month 53. '
    'Despite COVID impact and 2023 market headwinds, growth trajectory remained intact.',
    y=-0.12, fontsize=9,
)

# ── Bottom left: Segment contribution ────────────────────────────────
ax_bl = fig.add_axes([0.04, 0.04, 0.28, 0.42])
contribution_chart(
    ax_bl, segments, seg_contrib,
    palette='default', bar_height=0.6,
    show_values=True, value_fmt='{:.1f}%', sort=True,
)
ax_bl.set_title('ARR by Segment', loc='left',
                fontsize=10, fontweight='bold')
ax_bl.set_xlim(0, 55)
set_tick_style(ax_bl, labelsize=8)
despine(ax_bl, left=False)
add_note(ax_bl, 'Enterprise + Mid-Market = 72.7%',
         x=0.30, y=0.06, color='#374151', fontsize=7)

# ── Bottom center: Revenue bridge ────────────────────────────────────
ax_bc = fig.add_axes([0.36, 0.04, 0.30, 0.42])
waterfall_chart(
    ax_bc,
    labels=bridge_labels,
    values=bridge_values,
    positive_color='#16A34A', negative_color='#DC2626', total_color='#2563EB',
    total_indices=[0, 6],
    bar_width=0.55, show_values=True, value_fontsize=7,
)
ax_bc.set_title('2024 ARR Bridge', loc='left', fontsize=10, fontweight='bold')
ax_bc.set_ylabel('ARR ($K)'); format_thousands(ax_bc, 'y')
set_tick_style(ax_bc, labelsize=7)

# ── Bottom right: Forecast ────────────────────────────────────────────
ax_br = fig.add_axes([0.70, 0.04, 0.26, 0.42])
ax_br.plot(months[-18:], arr[-18:], color='#1D4ED8', linewidth=2.0, label='Historical')
shade_forecast_period(ax_br, months[-1], color='#FAF5FF', alpha=0.5,
                      linecolor='#7C3AED', linewidth=1.0)
add_forecast_region(ax_br, t_fore, fore_central, fore_lo, fore_hi,
                    central_color='#7C3AED', band_color='#C4B5FD',
                    central_linewidth=1.8, band_alpha=0.25,
                    linestyle='--', label='Forecast')
ax_br.set_title('2025 Forecast', loc='left', fontsize=10, fontweight='bold')
ax_br.set_ylabel('ARR ($K)'); format_thousands(ax_br, 'y')
ax_br.legend(frameon=False, fontsize=7)
despine(ax_br); set_grid(ax_br); set_tick_style(ax_br, labelsize=7)

plt.show()

---
## Summary — The Full Framework

```
DATA
 │
 ├── PERCEPTION          highlight_series, fade_series, focus_on_range
 │                       saliency_map_scatter, detect_and_mark_outliers
 │
 ├── CONTEXT             add_benchmark_band, add_target_line
 │                       shade_excess, shade_deficit, add_mean_line
 │
 ├── NARRATIVE           shade_regimes, add_event_markers
 │                       mark_turning_points, StoryFrame
 │
 ├── UNCERTAINTY         add_confidence_band, fan_chart
 │                       shade_forecast_period, add_forecast_region
 │
 ├── COGNITIVE LOAD      label_last_point, label_max/min
 │                       despine, format_thousands, apply_theme
 │
 ├── PROCESS             waterfall_chart, contribution_chart
 │                       decomposition_chart, stacked_contribution
 │
 ├── FRAMING CHECK       Honest Chart Checklist (07)
 │
 └── PIPELINE
      ├── StoryChart          signal + background + events
      ├── InsightChart        one focused insight
      ├── NarrativePipeline   full orchestration
      └── PipelineBuilder     custom fluent steps
                                        │
                                    INSIGHT
```

---

### Key Takeaway

> Visualization Intelligence không phải là biết dùng nhiều chart types.
> Nó là biết **tại sao** mỗi element tồn tại trong chart —
> mỗi màu sắc, mỗi đường, mỗi annotation đều phải phục vụ
> cho một mục đích duy nhất: **giúp người xem đưa ra quyết định tốt hơn**.
>
> ```
> Data ≠ Insight
> Insight = Data + Perception + Context + Narrative
> ```